# Sorting — Lesson

Sorting is one of the **most-used operations in all of programming**, and one of the most-tested concepts in interviews. Python's `sorted()` and `list.sort()` are powerful, fast, and surprisingly nuanced. This lesson covers everything you need:

- the two ways to sort (`sorted` vs `.sort()`)
- the `key` parameter — the single most important sorting concept in Python
- `lambda` for inline keys
- sorting by multiple fields
- ascending vs descending, including mixed orders
- stability and why it matters
- sorting strings, tuples, dicts, objects
- `operator.itemgetter` / `attrgetter` for performance
- sorting with custom comparators (`functools.cmp_to_key`)
- the algorithms behind it (Timsort) + complexity
- common interview patterns (top-k, sort-by-frequency, custom orderings, intervals, anagram grouping)
- how to use sorting in DSA problems
- gotchas and anti-patterns

### Why this matters
- **Interviews:** Half of "easy/medium" array problems become trivial after a sort. Top-K, intervals, anagrams, custom orderings — all sorting.
- **Backend:** Sorting query results, paginated APIs, leaderboards, deduplication.
- **Data engineering:** Sorting before grouping/joining, top-N reports, time-series ordering.

### What you'll learn
1. `sorted()` vs `list.sort()`
2. The `reverse=True` parameter
3. The `key` parameter (the workhorse)
4. Lambdas as keys
5. Sorting by multiple fields (tuple keys)
6. Mixed asc/desc with negation or two-pass sorts
7. Stability — what it is, why it matters
8. Sorting strings & lexicographic order
9. Sorting dicts (by key, by value)
10. Sorting objects (attrgetter, dataclasses)
11. `operator.itemgetter` and `attrgetter`
12. Custom comparators via `functools.cmp_to_key`
13. Timsort: the algorithm, time/space complexity
14. Comparison of classic sort algorithms (mergesort, quicksort, heapsort, etc.)
15. Common interview sorting patterns
16. Performance tips & gotchas

By the end you should be able to solve any sort-driven problem with the right tool in seconds.

---
## 1. `sorted()` vs `list.sort()`

Python gives you **two** ways to sort. Know the difference cold — interviewers ask.

| | `sorted(iterable)` | `list.sort()` |
|---|---|---|
| Returns | a **new** list | `None` (sorts in place) |
| Mutates input? | No | Yes |
| Works on | any iterable | only lists |
| When to use | when you need to keep the original, or input isn't a list | when you own the list and don't need the original |

In [ ]:
nums = [3, 1, 4, 1, 5, 9, 2, 6]

# sorted() — returns a new list, original untouched
out = sorted(nums)
print("sorted ->", out)
print("original ->", nums)

# .sort() — mutates in place, returns None
nums.sort()
print("after .sort() ->", nums)

# Common bug: assigning the return value of .sort()
broken = [3, 1, 2].sort()
print("broken ->", broken)   # None  ← classic interview gotcha

**Memorise:** `.sort()` returns `None`. Never write `nums = nums.sort()`.

`sorted()` works on **any iterable** — strings, tuples, sets, dicts, generators — and always returns a list.

In [ ]:
print(sorted("python"))            # list of chars
print(sorted((3, 1, 2)))            # tuple → list
print(sorted({3, 1, 2}))            # set → list
print(sorted({"b": 1, "a": 2}))     # dict → list of KEYS sorted
print(sorted(x*x for x in [3, 1, 2]))  # generator → list

---
## 2. `reverse=True` — Descending Order

Both `sorted()` and `.sort()` accept `reverse=True` for descending order.

In [ ]:
nums = [3, 1, 4, 1, 5, 9, 2, 6]
print(sorted(nums, reverse=True))   # descending

words = ["banana", "apple", "cherry"]
words.sort(reverse=True)
print(words)

**Tip:** for numeric data, `sorted(nums, reverse=True)` is equivalent to `sorted(nums, key=lambda x: -x)` — but `reverse=True` is clearer and faster.

---
## 3. The `key` Parameter — The Most Important Sorting Concept

`key` is a **function** that takes one item and returns the value to sort *by*. The original items are sorted, but the comparison uses the key.

```python
sorted(items, key=func)
```

This is how you sort by length, by case-insensitive value, by a tuple field, by an object attribute — anything.

In [ ]:
words = ["banana", "fig", "apple", "kiwi", "blueberry"]

# Sort by length (ascending)
print(sorted(words, key=len))

# Sort by length descending
print(sorted(words, key=len, reverse=True))

# Sort case-insensitively
mixed = ["banana", "Apple", "cherry", "ALMOND"]
print(sorted(mixed))                      # capital letters first (ASCII)
print(sorted(mixed, key=str.lower))       # case-insensitive

**Key idea:** the key function is called **once per item** (O(n)). The result is cached and used for all comparisons. This makes `key` both clean and fast — way better than a custom comparator.

---
## 4. `lambda` as a Key

`lambda` lets you write a one-line throwaway function inline. It's the most common way to express a key.

```python
sorted(items, key=lambda x: <expression using x>)
```

In [ ]:
# Sort by absolute value
nums = [-5, 2, -1, 3, -4]
print(sorted(nums, key=lambda x: abs(x)))

# Equivalent without lambda — pass abs directly
print(sorted(nums, key=abs))

# Sort tuples by the second element
pairs = [(1, 'b'), (2, 'a'), (3, 'c')]
print(sorted(pairs, key=lambda p: p[1]))

# Sort dicts (by a field) in a list
users = [{"name": "Alice", "age": 30},
         {"name": "Bob",   "age": 25},
         {"name": "Carol", "age": 35}]
print(sorted(users, key=lambda u: u["age"]))

**Tip:** if your `lambda` is just `lambda x: x.attr` or `lambda x: x[i]`, prefer `operator.attrgetter` / `itemgetter` — see §11. Faster and clearer.

---
## 5. Sorting by Multiple Fields (Tuple Keys)

Return a **tuple** from the key function. Python compares tuples element-by-element, so the first field is the primary sort key, the second the tiebreaker, and so on.

In [ ]:
people = [
    ("Alice",  30),
    ("Bob",    25),
    ("Alice",  22),
    ("Carol",  30),
    ("Bob",    40),
]

# Sort by name, then by age
print(sorted(people, key=lambda p: (p[0], p[1])))

# Sort by age, then by name (alphabetical tiebreak)
print(sorted(people, key=lambda p: (p[1], p[0])))

---
## 6. Mixed Ascending / Descending

`reverse=True` flips the **whole** sort. What if you need name ascending but age descending?

Three techniques:

### 6.1 Negate numeric fields
For numeric fields you can negate inside the key:

In [ ]:
# Name ASC, age DESC
print(sorted(people, key=lambda p: (p[0], -p[1])))

### 6.2 Two-pass sort (relies on stability)
Sort by the *secondary* key first, then by the *primary* key. Because Timsort is **stable**, items that compare equal on the primary key keep their secondary order.

In [ ]:
# Same result, multi-pass approach (works for non-numeric fields too)
sorted_by_age = sorted(people, key=lambda p: p[1], reverse=True)   # age DESC first
final = sorted(sorted_by_age, key=lambda p: p[0])                  # then name ASC
print(final)

### 6.3 Custom comparator (rare — for truly mixed orderings on non-numeric fields)
See §12 (`functools.cmp_to_key`).

---
## 7. Stability — Why It Matters

A sort is **stable** if equal elements keep their original relative order. Python's `sorted` and `.sort()` are **always stable** (since Python 2.3).

This is what makes the two-pass technique above work.

In [ ]:
# Stable: ties preserve insertion order
data = [("A", 3), ("B", 1), ("C", 3), ("D", 2)]
# Sort by the number — A and C both have 3, so A must stay before C.
print(sorted(data, key=lambda x: x[1]))

**Interview gotcha:** if you're asked "is Python's sort stable?" — yes. If you're asked "name a stable sort algorithm" — mergesort. Quicksort is **not** stable (in its standard form).

---
## 8. Sorting Strings — Lexicographic Order

Strings sort lexicographically (dictionary-like) using Unicode code points. This means:
- Digits come before uppercase letters.
- Uppercase letters come before lowercase letters.
- Shorter prefix wins on ties: `"app" < "apple"`.

In [ ]:
print(sorted(["banana", "Apple", "cherry", "20", "9"]))
# ['20', '9', 'Apple', 'banana', 'cherry']

# Note: '20' < '9' as strings! '2' < '9'. Sort numerically with int():
print(sorted(["20", "9", "100"], key=int))

### Sorting characters of a string
Returns a list — `join` it back if you need a string.

In [ ]:
s = "python"
chars_sorted = sorted(s)
print(chars_sorted)
print("".join(chars_sorted))   # 'hnopty'

---
## 9. Sorting Dicts

A dict has both keys and values. You usually want to sort the **items** by either.

In [ ]:
scores = {"alice": 92, "bob": 67, "carol": 85, "dave": 50}

# Sort dict by key (returns list of items)
print(sorted(scores.items()))

# Sort dict by value, ascending
print(sorted(scores.items(), key=lambda kv: kv[1]))

# Sort dict by value, descending
print(sorted(scores.items(), key=lambda kv: kv[1], reverse=True))

# Build a NEW dict in sorted order (Python 3.7+ dicts preserve insertion order)
sorted_by_score = dict(sorted(scores.items(), key=lambda kv: -kv[1]))
print(sorted_by_score)

---
## 10. Sorting Objects

For class instances, sort by attributes. Either use a lambda or `operator.attrgetter`.

In [ ]:
from dataclasses import dataclass

@dataclass
class User:
    name: str
    age: int

users = [User("Alice", 30), User("Bob", 25), User("Carol", 35)]

# Lambda
print(sorted(users, key=lambda u: u.age))

# attrgetter — cleaner & faster
from operator import attrgetter
print(sorted(users, key=attrgetter("age")))

# Sort by multiple attributes
print(sorted(users, key=attrgetter("age", "name")))

You can also implement `__lt__` on the class to make objects naturally comparable:

```python
@dataclass(order=True)
class User:
    age: int
    name: str
```
With `order=True`, dataclass generates `__lt__`, `__le__`, etc. automatically based on field order. Then `sorted(users)` works without a key.

---
## 11. `operator.itemgetter` and `attrgetter`

For sorting by a fixed index/field/attribute, prefer these over lambdas:

| Helper | Use for | Example |
|--------|---------|---------|
| `itemgetter(i)` | sequences (tuple/list) by index | `key=itemgetter(0)` |
| `itemgetter(k)` | dicts by key | `key=itemgetter("name")` |
| `itemgetter(0, 1)` | multiple keys → tuple | `key=itemgetter("age", "name")` |
| `attrgetter("a")` | objects by attribute | `key=attrgetter("age")` |
| `attrgetter("a.b")` | nested attribute | `key=attrgetter("address.city")` |

These are implemented in C, so they're noticeably faster than lambdas on large lists.

In [ ]:
from operator import itemgetter

records = [
    {"name": "Alice", "age": 30, "score": 92},
    {"name": "Bob",   "age": 25, "score": 67},
    {"name": "Carol", "age": 30, "score": 85},
]

# Sort by score descending
print(sorted(records, key=itemgetter("score"), reverse=True))

# Sort by age, then by name
print(sorted(records, key=itemgetter("age", "name")))

---
## 12. Custom Comparators — `functools.cmp_to_key`

Python 3 removed the `cmp` parameter. For the rare case where you need a true comparator (not a key), use `functools.cmp_to_key`. The comparator returns:
- **negative** if `a` should come before `b`
- **positive** if `a` should come after `b`
- **0** if equal

The classic interview problem this solves: **"Largest Number"** — concatenate numbers to form the largest possible string. Comparison is `b+a` vs `a+b`, which can't be expressed as a single key.

In [ ]:
from functools import cmp_to_key

def largest_number_cmp(a, b):
    # Want bigger concatenation first
    if a + b > b + a:
        return -1
    if a + b < b + a:
        return 1
    return 0

nums = ["3", "30", "34", "5", "9"]
result = sorted(nums, key=cmp_to_key(largest_number_cmp))
print("".join(result))   # "9534330"  — largest possible

**Use sparingly.** A comparator runs O(n log n) times — much slower than `key`. Prefer `key` whenever possible.

---
## 13. Timsort — Python's Sort Algorithm

Python's `sorted` and `.sort()` use **Timsort**, a hybrid of merge sort and insertion sort designed by Tim Peters in 2002.

| Property | Value |
|----------|-------|
| Time — best | **O(n)** (already-sorted input!) |
| Time — average / worst | **O(n log n)** |
| Space | **O(n)** (auxiliary, for merging) |
| Stable | **Yes** |
| In-place? | List `.sort()` is mostly in-place, but uses O(n) auxiliary space |
| Adaptive | Yes — exploits existing runs of sorted data |

**Why it's special:**
- Detects pre-existing sorted "runs" and merges them efficiently.
- Performs near-linearly on partially-sorted real-world data.
- Used by Java, Android, V8 (JavaScript), Rust, Swift — it's the de facto standard.

You don't implement Timsort in interviews, but **knowing it's stable and adaptive** is fair game.

---
## 14. Classic Sort Algorithms — Quick Reference

You won't usually implement these in Python (you'd use `sorted`), but interviewers love to ask about them.

| Algorithm | Best | Average | Worst | Space | Stable | In-Place |
|-----------|------|---------|-------|-------|--------|----------|
| Bubble    | O(n) | O(n²)   | O(n²) | O(1)  | Yes    | Yes |
| Insertion | O(n) | O(n²)   | O(n²) | O(1)  | Yes    | Yes |
| Selection | O(n²)| O(n²)   | O(n²) | O(1)  | No     | Yes |
| Merge     | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes | No |
| Quick     | O(n log n) | O(n log n) | O(n²) | O(log n) | No | Yes |
| Heap      | O(n log n) | O(n log n) | O(n log n) | O(1) | No | Yes |
| **Timsort** | O(n) | O(n log n) | O(n log n) | O(n) | Yes | (mostly) |
| Counting  | O(n+k) | O(n+k) | O(n+k) | O(k) | Yes | No |
| Radix     | O(nk) | O(nk) | O(nk) | O(n+k) | Yes | No |

**Key facts you should be able to recite:**
- Mergesort is **stable**, O(n log n) guaranteed, but needs O(n) extra space.
- Quicksort is **in-place**, average O(n log n), but worst O(n²) (when pivot is bad).
- Heapsort is **O(n log n) worst-case** and **in-place**, but **not stable**.
- Counting / radix sort are **O(n)** for small integer ranges — beat the O(n log n) lower bound for general sorts because they aren't comparison-based.

### Implementing common sorts (for whiteboard practice)

In [ ]:
def bubble_sort(a):
    a = a[:]
    n = len(a)
    for i in range(n):
        swapped = False
        for j in range(n - i - 1):
            if a[j] > a[j+1]:
                a[j], a[j+1] = a[j+1], a[j]
                swapped = True
        if not swapped:
            break
    return a

def insertion_sort(a):
    a = a[:]
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:
            a[j+1] = a[j]
            j -= 1
        a[j+1] = key
    return a

def merge_sort(a):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    left, right = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    out.extend(left[i:]); out.extend(right[j:])
    return out

def quick_sort(a):
    if len(a) <= 1:
        return a
    pivot = a[len(a)//2]
    less    = [x for x in a if x < pivot]
    equal   = [x for x in a if x == pivot]
    greater = [x for x in a if x > pivot]
    return quick_sort(less) + equal + quick_sort(greater)

xs = [3, 1, 4, 1, 5, 9, 2, 6, 5]
print(bubble_sort(xs))
print(insertion_sort(xs))
print(merge_sort(xs))
print(quick_sort(xs))

---
## 15. Common Interview Sorting Patterns

These come up over and over. Memorise the shape of each.

### 15.1 Sort then scan (unlock O(n log n) solutions)
Many problems collapse from O(n²) to O(n log n) once the input is sorted.

In [ ]:
# Are there duplicates? — easy after sorting
def has_duplicate(nums):
    s = sorted(nums)
    return any(s[i] == s[i-1] for i in range(1, len(s)))

print(has_duplicate([1, 2, 3, 4, 5]))    # False
print(has_duplicate([1, 2, 3, 2, 5]))    # True

### 15.2 Top K (and bottom K)

In [ ]:
# Top 3 largest
nums = [4, 7, 1, 9, 2, 5, 8, 3]
print(sorted(nums, reverse=True)[:3])

# Top 3 smallest
print(sorted(nums)[:3])

# For very large data prefer heapq.nlargest / nsmallest — O(n log k):
import heapq
print(heapq.nlargest(3, nums))
print(heapq.nsmallest(3, nums))

### 15.3 Sort by frequency

In [ ]:
from collections import Counter

s = "tree"
freq = Counter(s)
# Sort chars by frequency descending, then alphabetical for ties
out = sorted(freq.items(), key=lambda kv: (-kv[1], kv[0]))
print(out)

### 15.4 Group anagrams (sorted-letters key)

In [ ]:
from collections import defaultdict

def group_anagrams(words):
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    return list(groups.values())

print(group_anagrams(["eat","tea","tan","ate","nat","bat"]))

### 15.5 Merge intervals (sort by start)

In [ ]:
def merge_intervals(intervals):
    intervals = sorted(intervals, key=lambda x: x[0])
    out = []
    for start, end in intervals:
        if out and start <= out[-1][1]:
            out[-1][1] = max(out[-1][1], end)
        else:
            out.append([start, end])
    return out

print(merge_intervals([[1,3],[2,6],[8,10],[15,18]]))

### 15.6 Meeting rooms (sort + sweep)

In [ ]:
def can_attend_all(meetings):
    meetings.sort(key=lambda m: m[0])
    return all(meetings[i][0] >= meetings[i-1][1] for i in range(1, len(meetings)))

print(can_attend_all([[0,30],[5,10],[15,20]]))   # False
print(can_attend_all([[7,10],[2,4]]))            # True

### 15.7 Custom orderings

In [ ]:
# Sort strings by length, ties broken alphabetically
words = ["banana", "fig", "apple", "kiwi", "pear", "date"]
print(sorted(words, key=lambda w: (len(w), w)))

### 15.8 Sort by a derived score

In [ ]:
# Sort users by (name length DESC, then name ASC)
names = ["alice", "bob", "carol", "dave"]
print(sorted(names, key=lambda n: (-len(n), n)))

### 15.9 Reorder strings by a custom alphabet

In [ ]:
# Given a custom alphabet order, sort words by it
alphabet = "hlabcdefgijkmnopqrstuvwxyz"
order = {ch: i for i, ch in enumerate(alphabet)}
words = ["hello", "leetcode", "lab"]
print(sorted(words, key=lambda w: [order[c] for c in w]))

### 15.10 Largest-number / lexicographic concat (cmp_to_key)
See §12.

---
## 16. Performance Tips & Gotchas

### 16.1 Sorting once is faster than sorting many times
If you'll do many "is X in the top 10" lookups, sort once.

### 16.2 Avoid sorting if you only need the top K
- O(n log n) sort + slice is wasteful when K << n.
- `heapq.nlargest(k, xs)` is **O(n log k)** — use it.

### 16.3 Don't sort to find min/max
`min(xs)` / `max(xs)` is O(n). `sorted(xs)[0]` is O(n log n) — slower for no reason.

### 16.4 `key=` is computed once per item — exploit it
The result is cached. Even an expensive key function only runs n times, not n log n times.

### 16.5 `cmp_to_key` is slow — only use when no key works.

### 16.6 `.sort()` returns `None` (the #1 sorting bug)
```python
result = my_list.sort()   # WRONG — result is None
```

### 16.7 Sorting heterogeneous types raises `TypeError`
```python
sorted([1, "a", 2])   # TypeError in Python 3
```
Provide a key that produces comparable values: `key=str`, `key=lambda x: (type(x).__name__, x)`, etc.

### 16.8 `None` is not comparable
```python
sorted([3, None, 1])   # TypeError
```
Filter `None`s out, or use a key that handles them: `key=lambda x: (x is None, x)`.

### 16.9 Sorting strings of digits
`sorted(["10", "2", "1"])` → `['1', '10', '2']` (lexicographic). Use `key=int` for numeric order.

### 16.10 Mutating during sort
Don't change the list in another thread while sorting it. Don't mutate items in your `key` function.

---

## Summary

- `sorted(it)` returns new list; `.sort()` mutates in place and returns `None`.
- `key=` is the workhorse — passes one argument, returns the comparison value.
- `reverse=True` for descending.
- Tuple keys for multi-field sorts; negate numbers for mixed asc/desc.
- Python's sort is **stable** — exploit it with the two-pass technique.
- Use `operator.itemgetter` / `attrgetter` for clean, fast field-based sorts.
- `functools.cmp_to_key` only when a key truly can't express the comparison.
- Timsort: O(n log n) worst, O(n) on sorted data, stable, adaptive.
- Common patterns: top-K, sort-by-frequency, group anagrams, merge intervals, custom alphabet, largest-number.
- Use `heapq.nlargest/nsmallest` for top-K when K << n.

You now have everything. Open `drills.ipynb`, then `practice.ipynb`.